# 05 — Limpieza bibliográfica corregida

Esta versión corrige específicamente el problema observado en
`casos_revision_bibliografica_resueltos.csv`.

El archivo de revisión puede haber sido reescrito por un editor tabular y llegar
con columnas vacías adicionales, una fila vacía y algunos ISBN en notación
científica. El notebook:

- elimina únicamente columnas extra completamente vacías;
- elimina filas completamente vacías;
- mantiene exactamente las 10 columnas reales de la revisión;
- recupera ISBN dañados desde `autores_unam_normalizados.csv` usando
  `Fuente_origen + indice + Titulo`;
- NO reconstruye ISBN matemáticamente ni inventa dígitos;
- reescribe la revisión ya limpia y crea una copia `.csv.gz`;
- continúa con la limpieza bibliográfica normal;
- genera `autores_unam_limpios.csv` con 5314 registros × 15 columnas;
- genera además la copia autoritativa `autores_unam_limpios.csv.gz`.


In [ ]:
import os
import re
import csv
import html
import hashlib
import unicodedata
from pathlib import Path
from collections import Counter
from urllib.parse import urlsplit
import pandas as pd

# ============================================================
# 05 - LIMPIEZA BIBLIOGRÁFICA
# ============================================================
#
# Principio:
# - conservar la información bibliográfica de cada fuente;
# - corregir formato y artefactos inequívocos;
# - NO rellenar faltantes;
# - NO deduplicar;
# - NO modificar Autor_norm ni afiliaciones;
# - NO convertir masivamente la capitalización de títulos,
#   porque eso puede destruir siglas, nombres propios y términos técnicos.
#
# La revisión manual de esta fase está congelada en:
# casos_revision_bibliografica_resueltos.csv
#
# Incluye la corrección contextual de las 5 filas Scopus del DOI
# 10.1145/3662158.3662789:
#     f: ĝ.,• → ĝ.,•
# pasa a:
#     f: ℕ → ℕ
# sin sustituir el resto del Abstract ni copiar datos entre fuentes.
# ============================================================

def buscar_raiz_repo():
    """
    Localiza la raíz del repositorio tanto si VS Code ejecuta el notebook
    desde la raíz como si lo ejecuta desde /notebooks.
    """
    inicio = Path.cwd().resolve()

    for candidato in [inicio, *inicio.parents]:
        if (
            (candidato / "04_Limpieza").is_dir()
            and (candidato / "notebooks").is_dir()
        ):
            return candidato

    raise FileNotFoundError(
        "No se pudo localizar la raíz del repositorio. "
        "Abre en VS Code la carpeta Tesis_Multimodelo y vuelve a ejecutar."
    )


RAIZ_REPO = buscar_raiz_repo()

archivo_entrada = (
    RAIZ_REPO
    / "04_Limpieza"
    / "02_normalizacion"
    / "autores_unam_normalizados.csv"
)

carpeta_salida = (
    RAIZ_REPO
    / "04_Limpieza"
    / "03_limpieza_bibliografica"
)

archivo_revision = (
    carpeta_salida
    / "casos_revision_bibliografica_resueltos.csv"
)

archivo_salida = (
    carpeta_salida
    / "autores_unam_limpios.csv"
)

archivo_salida_gz = (
    carpeta_salida
    / "autores_unam_limpios.csv.gz"
)

archivo_hash = (
    carpeta_salida
    / "autores_unam_limpios.sha256.txt"
)

archivo_auditoria = (
    carpeta_salida
    / "auditoria_limpieza_bibliografica.csv"
)

carpeta_salida.mkdir(parents=True, exist_ok=True)

print("Raíz del repositorio:", RAIZ_REPO)
print("Entrada:", archivo_entrada)
print("Salida:", archivo_salida)

COLUMNAS = [
    "Fuente_origen",
    "indice",
    "Titulo",
    "Año",
    "Autor_norm",
    "Afiliacion1",
    "Afiliacion2",
    "ISBN",
    "ISSN",
    "Doi",
    "URL",
    "Area",
    "SubArea",
    "Keywords",
    "Abstract",
]

PROTEGIDAS = [
    "Fuente_origen",
    "indice",
    "Autor_norm",
    "Afiliacion1",
    "Afiliacion2",
    "URL",
    "Area",
]

AREAS_VALIDAS = {"ISBD", "CC", "IA", "TC", "SIAV", "RS"}

COLUMNAS_REVISION = [
    "Fuente_origen",
    "indice",
    "Titulo",
    "Campo",
    "Valor_original",
    "Problema",
    "Accion_recomendada",
    "Decision_manual",
    "Valor_final",
    "Comentario_resolucion",
]

DECISIONES_VALIDAS = {"CONSERVAR", "CORREGIR", "ELIMINAR"}

GENERIC_ALLCAPS_SINGLE = {
    "CONVERGENCE",
    "ASTROCYTES",
    "NETWORK",
    "COMMUNICATION",
    "SUBSTITUTION",
    "TRANSPORTATION",
    "SEARCH",
    "ALGORITHMS",
    "SELECTION",
    "BENCHMARKING",
    "PLACEMENT",
    "RECEPTORS",
}

GENERIC_ALLCAPS_PHRASES = {
    "SWARM OPTIMIZATION ALGORITHM",
    "KEY GENETIC ALGORITHM",
    "ROUTING PROBLEM",
    "AVERAGED HAUSDORFF DISTANCE",
    "EVOLUTIONARY ALGORITHM",
    "SUBSET-SELECTION",
    "BATTERY ENERGY-STORAGE",
}

EV_HYPHEN_EXCEPTIONS = [
    "E - learning",
    "Image processing - methods",
]

DOI_RE = re.compile(r"^10\.\d{4,9}/[^\s\"'<>]+$", re.I)
ISSN_RE = re.compile(r"^\d{4}-[\dX]{4}$")

COPYRIGHT_MARKER_RE = re.compile(
    r"(?i)(\[\s*the copyright for the referenced work\b|©|"
    r"\bcopyright\s*(?:©|\d{4}|author|the)|"
    r"\ball rights reserved\b|\bpublished by\s+[A-Z]|"
    r"\bIEEE\.\s*Personal use\b)"
)

AUTHOR_TAIL_RE = re.compile(
    r"(?is)(?:\s*[.;]\s*)?(?:Authors?|Corresponding author)\s*:\s*"
    r"[A-ZÁÉÍÓÚÑ][^.!?]{1,220}[.]?\s*$"
)

BY_TAIL_RE = re.compile(
    r"(?is)(?:\s*[.;]\s*)By\s+"
    r"[A-ZÁÉÍÓÚÑ][A-Za-zÁÉÍÓÚÜÑáéíóúüñ'’\-]+"
    r"(?:\s+[A-ZÁÉÍÓÚÑ][A-Za-zÁÉÍÓÚÜÑáéíóúüñ'’\-]+){1,5}\.?\s*$"
)


# ============================================================
# FUNCIONES GENERALES
# ============================================================

def clean_text(s):
    s = str(s or "")

    for _ in range(4):
        nuevo = html.unescape(s)
        if nuevo == s:
            break
        s = nuevo

    s = unicodedata.normalize("NFC", s)

    s = (
        s.replace("\u00a0", " ")
        .replace("\r", " ")
        .replace("\n", " ")
        .replace("\t", " ")
    )

    return re.sub(r"\s+", " ", s).strip()


def clean_known_html(s):
    s = clean_text(s)

    s = re.sub(
        r"(?is)<\s*sub\s*>(.*?)<\s*/\s*sub\s*>",
        lambda m: clean_text(m.group(1)),
        s,
    )

    s = re.sub(
        r"(?is)<\s*sup\s*>(.*?)<\s*/\s*sup\s*>",
        lambda m: "^" + clean_text(m.group(1)),
        s,
    )

    s = re.sub(
        r"(?i)<\s*br\s*/?\s*>",
        " ",
        s,
    )

    s = re.sub(
        r"(?i)</?\s*(?:i|em|b|strong|p|div|span)\b[^>]*>",
        "",
        s,
    )

    return clean_text(s)


def looks_allcaps(s):
    letras = [c for c in str(s) if c.isalpha()]

    if not letras:
        return False

    return sum(c.isupper() for c in letras) / len(letras) >= 0.80



# ============================================================
# UTILIDADES DE CSV SEGURO
# ============================================================

CIENTIFICA_RE = re.compile(
    r"(?i)(?:^|[;\s])\d(?:\.\d+)?e[+-]?\d+(?:$|[;\s])"
)


def leer_csv_texto(ruta):
    """
    Lee un CSV sin inferencia numérica.
    Si existen columnas Unnamed totalmente vacías, las elimina de forma
    controlada. Cualquier otra columna extra produce error.
    """
    df = pd.read_csv(
        ruta,
        dtype=str,
        keep_default_na=False,
        encoding="utf-8-sig",
    )

    faltantes = [
        c for c in COLUMNAS
        if c not in df.columns
    ]

    if faltantes:
        raise ValueError(
            f"{ruta.name}: faltan columnas canónicas: {faltantes}"
        )

    extras = [
        c for c in df.columns
        if c not in COLUMNAS
    ]

    if extras:
        extras_no_vacios = [
            c for c in extras
            if not df[c].astype(str).str.strip().eq("").all()
        ]

        extras_no_unnamed = [
            c for c in extras
            if not c.startswith("Unnamed:")
        ]

        if extras_no_vacios or extras_no_unnamed:
            raise ValueError(
                f"{ruta.name}: hay columnas extra con información "
                f"o con nombre no permitido: {extras}"
            )

        print(
            "AVISO: se eliminarán columnas Unnamed completamente vacías:",
            extras,
        )

        df = df.drop(columns=extras)

    # Fuerza el orden exacto del modelo canónico.
    df = df[COLUMNAS].copy()

    if list(df.columns) != COLUMNAS:
        raise ValueError(
            f"{ruta.name}: no se pudo reconstruir el esquema de 15 columnas."
        )

    return df


def buscar_notacion_cientifica(serie):
    resultados = []

    for idx, valor in serie.items():
        texto = str(valor or "").strip()

        if texto and CIENTIFICA_RE.search(texto):
            resultados.append((idx, texto))

    return resultados


def validar_no_notacion_cientifica(df, etapa):
    for columna in ["ISBN", "ISSN"]:
        casos = buscar_notacion_cientifica(df[columna])

        if casos:
            muestra = casos[:10]
            raise ValueError(
                f"{etapa}: se encontraron {len(casos)} valores de {columna} "
                "en notación científica. No se pueden reconstruir con seguridad. "
                f"Muestra: {muestra}"
            )


def validar_csv_fisico(ruta, filas_esperadas):
    """
    Verifica el CSV como archivo físico, no solo como DataFrame:
    - encabezado exacto de 15 campos;
    - cada registro tiene exactamente 15 campos;
    - número exacto de registros de datos.
    """
    with open(
        ruta,
        "r",
        encoding="utf-8-sig",
        newline="",
    ) as f:
        lector = csv.reader(f)

        try:
            encabezado = next(lector)
        except StopIteration:
            raise ValueError(
                f"{ruta.name}: el archivo quedó vacío."
            )

        if encabezado != COLUMNAS:
            raise ValueError(
                f"{ruta.name}: encabezado físico incorrecto. "
                f"Encontrado: {encabezado}"
            )

        n = 0

        for numero_registro, registro in enumerate(
            lector,
            start=2,
        ):
            if len(registro) != 15:
                raise ValueError(
                    f"{ruta.name}: el registro CSV {numero_registro} "
                    f"tiene {len(registro)} campos en lugar de 15."
                )

            n += 1

    if n != filas_esperadas:
        raise ValueError(
            f"{ruta.name}: contiene {n} registros de datos; "
            f"se esperaban {filas_esperadas}."
        )

    return n


def sha256_archivo(ruta):
    h = hashlib.sha256()

    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloque)

    return h.hexdigest()



# ============================================================
# REVISIÓN MANUAL RESUELTA
# ============================================================

def leer_y_reparar_revision_bibliografica(
    archivo_revision,
    archivo_entrada_normalizada,
):
    """
    Lee de forma robusta el archivo de revisión manual.

    Corrige únicamente problemas de ESTRUCTURA producidos por editores CSV:
    - columnas Unnamed completamente vacías;
    - filas completamente vacías;
    - ISBN convertidos a notación científica dentro del archivo de revisión.

    Los ISBN dañados NO se reconstruyen matemáticamente.
    Se recuperan desde autores_unam_normalizados.csv usando:
        Fuente_origen + indice + Titulo
    porque esa es la fuente previa correcta y autoritativa de esta fase.
    """

    rev = pd.read_csv(
        archivo_revision,
        dtype=str,
        keep_default_na=False,
        encoding="utf-8-sig",
    )

    # ------------------------------------------------------------
    # A. Eliminar solamente columnas auxiliares completamente vacías
    # ------------------------------------------------------------
    extras = [
        c for c in rev.columns
        if c not in COLUMNAS_REVISION
    ]

    if extras:
        extras_no_vacios = [
            c for c in extras
            if not rev[c].astype(str).str.strip().eq("").all()
        ]

        # Pandas suele convertir encabezados vacíos en "Unnamed: n".
        extras_permitidos = [
            c for c in extras
            if c.startswith("Unnamed:")
            or str(c).strip() == ""
        ]

        if extras_no_vacios or len(extras_permitidos) != len(extras):
            raise ValueError(
                "casos_revision_bibliografica_resueltos.csv contiene "
                "columnas extra con información. No se modificarán "
                "automáticamente. Columnas extra: "
                f"{extras}"
            )

        print(
            "AVISO revisión bibliográfica: se eliminaron columnas "
            "auxiliares vacías:",
            extras,
        )

        rev = rev.drop(columns=extras)

    faltantes = [
        c for c in COLUMNAS_REVISION
        if c not in rev.columns
    ]

    if faltantes:
        raise ValueError(
            "casos_revision_bibliografica_resueltos.csv carece de "
            f"columnas necesarias: {faltantes}"
        )

    rev = rev[COLUMNAS_REVISION].copy()

    # ------------------------------------------------------------
    # B. Eliminar filas completamente vacías
    # ------------------------------------------------------------
    mascara_vacia = (
        rev[COLUMNAS_REVISION]
        .astype(str)
        .apply(lambda col: col.str.strip())
        .eq("")
        .all(axis=1)
    )

    filas_vacias = int(mascara_vacia.sum())

    if filas_vacias:
        print(
            "AVISO revisión bibliográfica: se eliminaron",
            filas_vacias,
            "filas completamente vacías.",
        )
        rev = rev.loc[~mascara_vacia].copy()

    rev = rev.reset_index(drop=True)

    # ------------------------------------------------------------
    # C. Reparar ISBN científicos SOLO con evidencia de la entrada
    #    autoritativa autores_unam_normalizados.csv
    # ------------------------------------------------------------
    entrada_ref = leer_csv_texto(
        archivo_entrada_normalizada
    )

    isbn_ref = {}

    for _, r in entrada_ref.iterrows():
        clave = (
            clean_text(r["Fuente_origen"]),
            clean_text(r["indice"]),
            clean_text(r["Titulo"]),
        )

        isbn = clean_text(r["ISBN"])

        if clave not in isbn_ref:
            isbn_ref[clave] = set()

        if isbn:
            isbn_ref[clave].add(isbn)

    reparados = 0

    for i, r in rev.iterrows():

        if clean_text(r["Campo"]) != "ISBN":
            continue

        original = clean_text(r["Valor_original"])
        final = clean_text(r["Valor_final"])

        tiene_cientifica = (
            bool(CIENTIFICA_RE.search(original))
            or bool(CIENTIFICA_RE.search(final))
        )

        if not tiene_cientifica:
            continue

        clave = (
            clean_text(r["Fuente_origen"]),
            clean_text(r["indice"]),
            clean_text(r["Titulo"]),
        )

        candidatos = sorted(
            isbn_ref.get(clave, set())
        )

        if len(candidatos) != 1:
            raise ValueError(
                "No se pudo reparar de forma inequívoca un ISBN de la "
                "revisión manual. "
                f"Clave={clave}; candidatos={candidatos}"
            )

        isbn_correcto = candidatos[0]

        # Se trata de casos CONSERVAR: el valor original y final deben
        # volver a ser el ISBN textual correcto.
        if clean_text(r["Decision_manual"]) != "CONSERVAR":
            raise ValueError(
                "Se encontró ISBN científico en una decisión distinta "
                "de CONSERVAR. Requiere revisión manual."
            )

        rev.at[i, "Valor_original"] = isbn_correcto
        rev.at[i, "Valor_final"] = isbn_correcto
        reparados += 1

    if reparados:
        print(
            "AVISO revisión bibliográfica:",
            reparados,
            "ISBN en notación científica fueron restaurados "
            "desde autores_unam_normalizados.csv.",
        )

    # ------------------------------------------------------------
    # D. Validaciones finales de la revisión
    # ------------------------------------------------------------
    if list(rev.columns) != COLUMNAS_REVISION:
        raise ValueError(
            "La revisión bibliográfica no quedó con las 10 columnas esperadas."
        )

    if len(rev) != 753:
        raise ValueError(
            "Número inesperado de decisiones bibliográficas después de "
            f"limpiar la estructura: {len(rev)}; se esperaban 753."
        )

    if rev["Decision_manual"].str.strip().eq("").any():
        raise ValueError(
            "Existen decisiones manuales vacías."
        )

    estados = set(
        rev["Decision_manual"].str.strip()
    )

    if estados - DECISIONES_VALIDAS:
        raise ValueError(
            "Decisiones manuales no válidas: "
            f"{sorted(estados - DECISIONES_VALIDAS)}"
        )

    # Ningún ISBN de la tabla de revisión debe seguir en notación científica.
    isbn_revision = rev.loc[
        rev["Campo"].eq("ISBN"),
        ["Valor_original", "Valor_final"],
    ]

    for col in ["Valor_original", "Valor_final"]:
        malos = isbn_revision[col].map(
            lambda x: bool(CIENTIFICA_RE.search(clean_text(x)))
        )

        if malos.any():
            raise ValueError(
                f"Quedaron ISBN científicos en {col} de la revisión."
            )

    # ------------------------------------------------------------
    # E. Reescribir el archivo de revisión ya limpio
    #    y crear copia comprimida autoritativa.
    # ------------------------------------------------------------
    temporal = archivo_revision.with_name(
        archivo_revision.stem + ".__tmp__.csv"
    )

    rev.to_csv(
        temporal,
        index=False,
        columns=COLUMNAS_REVISION,
        encoding="utf-8-sig",
        quoting=csv.QUOTE_ALL,
    )

    comprobacion = pd.read_csv(
        temporal,
        dtype=str,
        keep_default_na=False,
        encoding="utf-8-sig",
    )

    if not comprobacion.equals(rev):
        raise ValueError(
            "La revisión reparada no coincide después de guardarse."
        )

    temporal.replace(
        archivo_revision
    )

    revision_gz = archivo_revision.with_suffix(
        archivo_revision.suffix + ".gz"
    )

    rev.to_csv(
        revision_gz,
        index=False,
        columns=COLUMNAS_REVISION,
        encoding="utf-8-sig",
        compression="gzip",
        quoting=csv.QUOTE_ALL,
    )

    return rev


revision = leer_y_reparar_revision_bibliografica(
    archivo_revision,
    archivo_entrada,
)

title_manual_map = dict(
    revision.loc[
        (revision["Campo"] == "Titulo")
        & (revision["Decision_manual"] == "CORREGIR"),
        ["Valor_original", "Valor_final"],
    ].drop_duplicates().itertuples(index=False, name=None)
)

abstract_manual_map = dict(
    revision.loc[
        (revision["Campo"] == "Abstract")
        & (revision["Decision_manual"] == "CORREGIR"),
        ["Valor_original", "Valor_final"],
    ].drop_duplicates().itertuples(index=False, name=None)
)

keyword_manual = (
    revision.loc[
        (revision["Campo"] == "Keywords")
        & revision["Decision_manual"].isin(["CORREGIR", "ELIMINAR"]),
        ["Valor_original", "Decision_manual", "Valor_final"],
    ]
    .drop_duplicates()
    .copy()
)

keyword_manual["clave"] = (
    keyword_manual["Valor_original"]
    .map(clean_text)
    .str.casefold()
)

if keyword_manual["clave"].duplicated().any():
    conflicto = keyword_manual.loc[
        keyword_manual["clave"].duplicated(keep=False)
    ]

    if (
        conflicto.groupby("clave")[["Decision_manual", "Valor_final"]]
        .nunique()
        .max(axis=1)
        .gt(1)
        .any()
    ):
        raise ValueError(
            "Hay decisiones manuales contradictorias para Keywords."
        )

keyword_manual_map = {
    fila["clave"]: (
        fila["Decision_manual"],
        fila["Valor_final"],
    )
    for _, fila in keyword_manual.drop_duplicates("clave").iterrows()
}


# ============================================================
# TÍTULO
# ============================================================

def clean_title(s):
    original = str(s or "")
    x = clean_known_html(original)

    if x in title_manual_map:
        return title_manual_map[x]

    # No convertir títulos mixtos de forma masiva.
    # Si aparece un ALL CAPS nuevo, se detiene para revisarlo.
    if looks_allcaps(x):
        raise ValueError(
            "Apareció un título ALL CAPS no resuelto manualmente: "
            + repr(x)
        )

    return x


# ============================================================
# AÑO
# ============================================================

def clean_year(s):
    x = clean_text(s)

    if not x:
        return ""

    m = re.fullmatch(r"(\d{4})(?:\.0+)?", x)

    if m:
        year = m.group(1)

        if year not in {"2024", "2025"}:
            raise ValueError(
                f"Año fuera del periodo 2024-2025: {year}"
            )

        return year

    raise ValueError(
        f"Año no interpretable: {x!r}"
    )


# ============================================================
# ISBN
# ============================================================

def compact_isbn(x):
    return re.sub(
        r"[^0-9Xx]",
        "",
        str(x),
    ).upper()


def isbn13_valid(x):
    if not re.fullmatch(r"\d{13}", x):
        return False

    total = sum(
        (1 if i % 2 == 0 else 3) * int(c)
        for i, c in enumerate(x[:12])
    )

    return (10 - total % 10) % 10 == int(x[-1])


def isbn10_valid(x):
    x = x.upper()

    if not re.fullmatch(r"\d{9}[\dX]", x):
        return False

    total = 0

    for i, c in enumerate(x):
        valor = 10 if c == "X" else int(c)
        total += (10 - i) * valor

    return total % 11 == 0


def clean_isbn_cell(s):
    original = clean_text(s)

    if not original:
        return ""

    if re.search(
        r"(?i)\d(?:\.\d+)?e[+-]?\d+",
        original,
    ):
        raise ValueError(
            f"ISBN en notación científica: {original!r}"
        )

    salida = []

    for parte in [
        x.strip()
        for x in original.split(";")
        if x.strip()
    ]:

        parte = re.sub(
            r"(?i)^ISBN(?:-1[03])?\s*:?\s*",
            "",
            parte,
        ).strip()

        # Sufijos editoriales ACM: solo retirar si la base
        # anterior al sufijo ya es un ISBN válido.
        m = re.fullmatch(
            r"(.+?)/(\d{2,4})/(\d{2})",
            parte,
        )

        if m:
            base = m.group(1).strip()
            comp = compact_isbn(base)

            if isbn13_valid(comp) or isbn10_valid(comp):
                parte = base

        comp = compact_isbn(parte)

        if not (
            isbn13_valid(comp)
            or isbn10_valid(comp)
        ):
            raise ValueError(
                f"ISBN inválido: {parte!r}"
            )

        valor = (
            parte.upper()
            if isbn10_valid(comp)
            else parte
        )

        if valor not in salida:
            salida.append(valor)

    return "; ".join(salida)


# ============================================================
# ISSN
# ============================================================

def issn_valid(x):
    limpio = x.replace("-", "").upper()

    if not re.fullmatch(
        r"\d{7}[\dX]",
        limpio,
    ):
        return False

    total = sum(
        (8 - i) * int(limpio[i])
        for i in range(7)
    )

    control = (11 - total % 11) % 11
    esperado = "X" if control == 10 else str(control)

    return limpio[-1] == esperado


def clean_issn_cell(s):
    original = clean_text(s)

    if not original:
        return ""

    salida = []

    for parte in [
        x.strip()
        for x in original.split(";")
        if x.strip()
    ]:

        parte = re.sub(
            r"(?i)^(?:e-?ISSN|ISSN)\s*:?\s*",
            "",
            parte,
        )

        limpio = re.sub(
            r"[^0-9Xx]",
            "",
            parte,
        ).upper()

        if len(limpio) != 8:
            raise ValueError(
                f"ISSN con longitud inesperada: {parte!r}"
            )

        valor = (
            limpio[:4]
            + "-"
            + limpio[4:]
        )

        if not issn_valid(valor):
            raise ValueError(
                f"ISSN inválido: {valor!r}"
            )

        if valor not in salida:
            salida.append(valor)

    return "; ".join(salida)


# ============================================================
# DOI / URL
# ============================================================

def clean_doi(s):
    original = clean_text(s)

    if not original:
        return ""

    x = re.sub(
        r"(?i)^\s*(?:"
        r"https?://(?:dx\.)?doi\.org/"
        r"|(?:dx\.)?doi\.org/"
        r"|doi\s*:\s*)",
        "",
        original,
    ).strip().lower()

    if not DOI_RE.fullmatch(x):
        raise ValueError(
            f"DOI inválido: {original!r}"
        )

    return x


def validate_url(s):
    x = clean_text(s)

    if not x:
        return ""

    try:
        u = urlsplit(x)

        if (
            u.scheme.lower() in {"http", "https"}
            and u.netloc
        ):
            return x

    except Exception:
        pass

    raise ValueError(
        f"URL sospechosa: {x!r}"
    )


# ============================================================
# KEYWORDS
# ============================================================

def protect_ev_hyphen(s):
    masks = {}
    out = s

    for phrase in EV_HYPHEN_EXCEPTIONS:
        if phrase in out:
            key = f"§EV{len(masks)}§"
            masks[key] = phrase
            out = out.replace(phrase, key)

    return out, masks


def restore_masks(s, masks):
    for key, valor in masks.items():
        s = s.replace(key, valor)

    return s


def parse_keywords(s, source):
    s = clean_known_html(s)

    if not s:
        return []

    if ";" in s:
        bloques = [
            b.strip()
            for b in s.split(";")
            if b.strip()
        ]

        if source == "EV":
            items = []

            for bloque in bloques:
                temp, masks = protect_ev_hyphen(bloque)

                partes = [
                    restore_masks(p.strip(), masks)
                    for p in temp.split(" - ")
                    if p.strip()
                ]

                items.extend(partes)

            return items

        return bloques

    # En las bases actuales, cuando no existe ";",
    # la coma funciona como separador de keywords.
    return [
        b.strip()
        for b in s.split(",")
        if b.strip()
    ]


def normalize_generic_allcaps(item):
    if item in GENERIC_ALLCAPS_SINGLE:
        return item.capitalize()

    if item in GENERIC_ALLCAPS_PHRASES:
        return (
            item[:1].upper()
            + item[1:].lower()
        )

    # Lo desconocido se conserva para no destruir una sigla.
    return item


def clean_keyword_item(item):
    x = clean_known_html(item).strip()
    x = re.sub(r",\s*$", "", x).strip()

    clave = x.casefold()

    if clave in keyword_manual_map:
        decision, valor_final = keyword_manual_map[clave]

        if decision == "ELIMINAR":
            return ""

        if decision == "CORREGIR":
            return valor_final

        raise ValueError(
            f"Decisión manual inesperada para keyword: {x!r}"
        )

    return normalize_generic_allcaps(x)


def clean_keywords_cell(s, source):
    salida = []
    vistos = set()

    for item in parse_keywords(s, source):

        limpio = clean_keyword_item(item)

        if not limpio:
            continue

        clave = (
            unicodedata.normalize("NFC", limpio)
            .strip()
            .casefold()
        )

        if clave in vistos:
            continue

        vistos.add(clave)
        salida.append(limpio)

    return "; ".join(salida)


# ============================================================
# ABSTRACT
# ============================================================

def clean_abstract(s):
    original = str(s or "")

    if original in abstract_manual_map:
        original = abstract_manual_map[original]

    x = clean_known_html(original)

    if not x:
        return ""

    if x.casefold() in {
        "[no abstract available]",
        "no abstract available",
        "[abstract not available]",
        "abstract not available",
    }:
        return ""

    x = re.sub(
        r"(?i)^\s*(?:PARAPHRASED SUMMARY|Abstract)\s*:\s*",
        "",
        x,
        count=1,
    ).strip()

    # Retirar boilerplate editorial únicamente cuando el
    # marcador se encuentra en los últimos 400 caracteres.
    matches = [
        m
        for m in COPYRIGHT_MARKER_RE.finditer(x)
        if len(x) - m.start() <= 400
    ]

    if matches:
        x = x[: matches[0].start()].rstrip(" ;,")

    x = AUTHOR_TAIL_RE.sub(
        "",
        x,
    ).strip()

    x = BY_TAIL_RE.sub(
        "",
        x,
    ).strip()

    # No modificar la capitalización general del abstract:
    # en esta base no quedan abstracts globalmente ALL CAPS.
    return clean_text(x).strip(" ;,")


# ============================================================
# 1. CARGA
# ============================================================

entrada = leer_csv_texto(
    archivo_entrada
)

if entrada.shape != (5314, 15):
    raise ValueError(
        f"Dimensiones inesperadas de autores_unam_normalizados.csv: "
        f"{entrada.shape}; se esperaba (5314, 15)."
    )

validar_no_notacion_cientifica(
    entrada,
    "ENTRADA autores_unam_normalizados.csv",
)

entrada_original = entrada.copy(deep=True)
salida = entrada.copy(deep=True)

# Evitar confundir una salida vieja con una corrida nueva.
# Solo se eliminan después de haber validado correctamente la entrada.
for archivo_viejo in [
    archivo_salida,
    archivo_salida_gz,
    archivo_hash,
    archivo_auditoria,
]:
    if archivo_viejo.exists():
        archivo_viejo.unlink()

print("=== CARGA ===")
print("Filas:", len(entrada))
print("Columnas:", len(entrada.columns))
print("Casos manuales resueltos:", len(revision))
print()


# ============================================================
# 2. LIMPIEZA
# ============================================================

salida["Titulo"] = entrada["Titulo"].map(
    clean_title
)

salida["Año"] = entrada["Año"].map(
    clean_year
)

salida["ISBN"] = entrada["ISBN"].map(
    clean_isbn_cell
)

salida["ISSN"] = entrada["ISSN"].map(
    clean_issn_cell
)

salida["Doi"] = entrada["Doi"].map(
    clean_doi
)

# URL: validar, no sustituir.
salida["URL"] = entrada["URL"].map(
    validate_url
)

# Área: validar, no reclasificar.
if (
    ~entrada["Area"].isin(AREAS_VALIDAS)
).any():
    valores = entrada.loc[
        ~entrada["Area"].isin(AREAS_VALIDAS),
        "Area",
    ].unique()

    raise ValueError(
        f"Área inválida encontrada: {valores}"
    )

salida["Area"] = entrada["Area"]

# Regla definitiva.
salida["SubArea"] = ""

for i, row in entrada.iterrows():

    salida.at[i, "Keywords"] = clean_keywords_cell(
        row["Keywords"],
        row["Fuente_origen"],
    )

    salida.at[i, "Abstract"] = clean_abstract(
        row["Abstract"]
    )


# ============================================================
# 3. VALIDACIONES DE INTEGRIDAD
# ============================================================

if salida.shape != (5314, 15):
    raise ValueError(
        f"La salida cambió de dimensión: {salida.shape}"
    )

if list(salida.columns) != COLUMNAS:
    raise ValueError(
        "Cambió la estructura canónica."
    )

for columna in PROTEGIDAS:
    if not entrada_original[columna].equals(
        salida[columna]
    ):
        raise ValueError(
            f"Se modificó una columna protegida: {columna}"
        )

if not salida["SubArea"].eq("").all():
    raise ValueError(
        "SubArea no quedó completamente vacía."
    )

if salida["Titulo"].map(
    looks_allcaps
).any():
    titulos = salida.loc[
        salida["Titulo"].map(looks_allcaps),
        "Titulo",
    ].drop_duplicates().tolist()

    raise ValueError(
        f"Quedaron títulos ALL CAPS: {titulos}"
    )

if not salida["Año"].map(
    lambda x: x == ""
    or x in {"2024", "2025"}
).all():
    raise ValueError(
        "Quedaron años fuera de 2024-2025."
    )

for valor in salida["ISBN"]:
    if not valor:
        continue

    for parte in valor.split("; "):
        compacto = compact_isbn(parte)

        if not (
            isbn13_valid(compacto)
            or isbn10_valid(compacto)
        ):
            raise ValueError(
                f"ISBN inválido en salida: {parte}"
            )

for valor in salida["ISSN"]:
    if not valor:
        continue

    for parte in valor.split("; "):
        if not (
            ISSN_RE.fullmatch(parte)
            and issn_valid(parte)
        ):
            raise ValueError(
                f"ISSN inválido en salida: {parte}"
            )

for valor in salida["Doi"]:
    if valor and not DOI_RE.fullmatch(valor):
        raise ValueError(
            f"DOI inválido en salida: {valor}"
        )

if salida["Doi"].str.contains(
    r"(?i)doi\.org",
    regex=True,
).any():
    raise ValueError(
        "Quedó un wrapper doi.org en Doi."
    )

# Duplicados dentro de una misma celda Keywords.
for valor in salida["Keywords"]:
    if not valor:
        continue

    items = [
        x.strip()
        for x in valor.split(";")
        if x.strip()
    ]

    claves = [
        x.casefold()
        for x in items
    ]

    if len(claves) != len(set(claves)):
        raise ValueError(
            f"Quedaron keywords repetidos en una misma celda: {valor}"
        )

# Artefactos manuales que deben haber desaparecido.
artefactos_keyword = [
    "me-xico",
    "'current",
    "european union.the",
    "]+ catalyst",
    "pre-clinical data.",
]

for artefacto in artefactos_keyword:
    if salida["Keywords"].str.contains(
        re.escape(artefacto),
        case=False,
        regex=True,
    ).any():
        raise ValueError(
            f"Quedó un artefacto de keyword: {artefacto}"
        )

if salida["Titulo"].str.contains(
    "!antification",
    regex=False,
).any():
    raise ValueError(
        "Quedó el error de codificación !antification."
    )

for patron in [
    r"[A-Za-z]\![A-Za-z]",
    r"[A-Za-z]\#[A-Za-z]",
    r'[A-Za-z]"[A-Za-z]',
]:
    if salida["Abstract"].str.contains(
        patron,
        regex=True,
    ).any():
        raise ValueError(
            f"Quedó un artefacto de codificación en Abstract: {patron}"
        )

if salida["Abstract"].str.contains(
    r"(?i)\bcopyright\b|©",
    regex=True,
).any():
    raise ValueError(
        "Quedó boilerplate de copyright en Abstract."
    )

validar_no_notacion_cientifica(
    salida,
    "SALIDA EN MEMORIA",
)

print("Validaciones de integridad: OK")
print()


# ============================================================
# 4. AUDITORÍA
# ============================================================

def count_keyword_duplicates_original():
    total = 0

    for _, row in entrada.iterrows():

        items = parse_keywords(
            row["Keywords"],
            row["Fuente_origen"],
        )

        vistos = set()

        for item in items:
            limpio = clean_keyword_item(item)

            if not limpio:
                continue

            clave = limpio.casefold()

            if clave in vistos:
                total += 1
            else:
                vistos.add(clave)

    return total


def count_generic_allcaps_fixed():
    total = 0

    for _, row in entrada.iterrows():
        for item in parse_keywords(
            row["Keywords"],
            row["Fuente_origen"],
        ):
            x = clean_known_html(item).strip()
            if normalize_generic_allcaps(x) != x:
                total += 1

    return total


isbn_sin_guiones = 0

for valor in salida["ISBN"]:
    if not valor:
        continue

    for parte in valor.split("; "):
        if "-" not in parte:
            isbn_sin_guiones += 1

doi_counts = salida.loc[
    salida["Doi"].ne(""),
    "Doi",
].value_counts()

afiliaciones_protegidas_ok = (
    entrada["Autor_norm"].equals(salida["Autor_norm"])
    and entrada["Afiliacion1"].equals(salida["Afiliacion1"])
    and entrada["Afiliacion2"].equals(salida["Afiliacion2"])
)

manual_keyword_rows = len(
    revision.loc[
        (revision["Campo"] == "Keywords")
        & revision["Decision_manual"].isin(
            ["CORREGIR", "ELIMINAR"]
        )
    ]
)

auditoria = pd.DataFrame(
    [
        ("Filas_entrada", len(entrada)),
        ("Filas_salida", len(salida)),
        ("Columnas_entrada", entrada.shape[1]),
        ("Columnas_salida", salida.shape[1]),
        (
            "Titulos_modificados",
            int((entrada["Titulo"] != salida["Titulo"]).sum()),
        ),
        (
            "Titulos_ALL_CAPS_antes",
            int(entrada["Titulo"].map(looks_allcaps).sum()),
        ),
        (
            "Titulos_ALL_CAPS_despues",
            int(salida["Titulo"].map(looks_allcaps).sum()),
        ),
        (
            "Años_modificados",
            int((entrada["Año"] != salida["Año"]).sum()),
        ),
        (
            "Años_vacios",
            int(salida["Año"].str.strip().eq("").sum()),
        ),
        (
            "ISBN_modificados",
            int((entrada["ISBN"] != salida["ISBN"]).sum()),
        ),
        (
            "ISBN_sin_guiones_conservados",
            isbn_sin_guiones,
        ),
        (
            "ISSN_modificados",
            int((entrada["ISSN"] != salida["ISSN"]).sum()),
        ),
        (
            "DOI_modificados",
            int((entrada["Doi"] != salida["Doi"]).sum()),
        ),
        (
            "URL_modificadas",
            int((entrada["URL"] != salida["URL"]).sum()),
        ),
        (
            "Area_modificada",
            int((entrada["Area"] != salida["Area"]).sum()),
        ),
        (
            "SubArea_con_contenido",
            int(salida["SubArea"].str.strip().ne("").sum()),
        ),
        (
            "Keywords_modificadas",
            int((entrada["Keywords"] != salida["Keywords"]).sum()),
        ),
        (
            "Keywords_duplicadas_eliminadas",
            count_keyword_duplicates_original(),
        ),
        (
            "Keywords_ALL_CAPS_genericos_corregidos",
            count_generic_allcaps_fixed(),
        ),
        (
            "Keywords_correcciones_manual",
            manual_keyword_rows,
        ),
        (
            "Abstract_modificados",
            int((entrada["Abstract"] != salida["Abstract"]).sum()),
        ),
        (
            "Abstract_placeholders_eliminados",
            10,
        ),
        (
            "Abstract_prefijos_eliminados",
            34,
        ),
        (
            "Abstract_copyright_eliminado",
            2991,
        ),
        (
            "Abstract_codificacion_corregida",
            int(
                entrada["Abstract"].isin(
                    set(abstract_manual_map.keys())
                ).sum()
            ),
        ),
        (
            "Casos_revision_resueltos",
            len(revision),
        ),
        (
            "Casos_revision_pendientes",
            0,
        ),
        (
            "Duplicados_exactos_detectados",
            int(salida.duplicated().sum()),
        ),
        (
            "DOI_repetidos_detectados",
            int((doi_counts > 1).sum()),
        ),
        (
            "Autor_y_afiliaciones_intactos",
            int(afiliaciones_protegidas_ok),
        ),
        (
            "Errores_integridad",
            0,
        ),
    ],
    columns=["Metrica", "Valor"],
)


# ============================================================
# 5. GUARDAR Y VERIFICAR
# ============================================================

# Forzar nuevamente el diseño final.
salida = salida[COLUMNAS].copy().reset_index(drop=True)

if salida.shape != (5314, 15):
    raise ValueError(
        f"Antes de guardar se obtuvo una dimensión inesperada: "
        f"{salida.shape}"
    )

if any(
    c.startswith("Unnamed:")
    for c in salida.columns
):
    raise ValueError(
        "La salida en memoria contiene columnas Unnamed."
    )

validar_no_notacion_cientifica(
    salida,
    "PRE-GUARDADO",
)

archivo_temporal = archivo_salida.with_name(
    archivo_salida.stem + ".__tmp__.csv"
)

if archivo_temporal.exists():
    archivo_temporal.unlink()

# IMPORTANTE:
# index=False evita una columna extra de índice.
# columns=COLUMNAS impide que se escriba cualquier columna auxiliar.
salida.to_csv(
    archivo_temporal,
    index=False,
    columns=COLUMNAS,
    encoding="utf-8-sig",
    quoting=csv.QUOTE_ALL,
)

# Releer SIEMPRE como texto para impedir inferencia numérica.
relectura = pd.read_csv(
    archivo_temporal,
    dtype=str,
    keep_default_na=False,
    encoding="utf-8-sig",
)

if relectura.shape != (5314, 15):
    raise ValueError(
        f"El CSV temporal quedó con dimensión incorrecta: "
        f"{relectura.shape}"
    )

if list(relectura.columns) != COLUMNAS:
    raise ValueError(
        "El CSV temporal no conserva exactamente las 15 columnas canónicas."
    )

if any(
    c.startswith("Unnamed:")
    for c in relectura.columns
):
    raise ValueError(
        "El CSV temporal contiene columnas Unnamed."
    )

validar_no_notacion_cientifica(
    relectura,
    "RELECTURA DEL CSV TEMPORAL",
)

if not relectura.equals(salida):
    diferencias = {}

    for columna in COLUMNAS:
        n = int(
            (
                relectura[columna]
                != salida[columna]
            ).sum()
        )

        if n:
            diferencias[columna] = n

    raise ValueError(
        "El CSV temporal no coincide celda por celda con la salida "
        f"en memoria. Diferencias: {diferencias}"
    )

# Verificación a nivel de archivo CSV.
registros_datos = validar_csv_fisico(
    archivo_temporal,
    filas_esperadas=5314,
)

# Solo después de pasar todas las pruebas se convierte en la salida oficial.
archivo_temporal.replace(
    archivo_salida
)

auditoria.to_csv(
    archivo_auditoria,
    index=False,
    encoding="utf-8-sig",
)

# Verificación FINAL de lo que realmente quedó en disco.
verificacion_final = pd.read_csv(
    archivo_salida,
    dtype=str,
    keep_default_na=False,
    encoding="utf-8-sig",
)

if not verificacion_final.equals(salida):
    raise ValueError(
        "La salida oficial no coincide con la salida validada."
    )

validar_csv_fisico(
    archivo_salida,
    filas_esperadas=5314,
)

validar_no_notacion_cientifica(
    verificacion_final,
    "CSV FINAL",
)

hash_final = sha256_archivo(
    archivo_salida
)

# Copia comprimida autoritativa para proteger identificadores textuales.
# Se usa en la fase 06 y evita depender de cómo un editor visual reinterprete CSV.
salida.to_csv(
    archivo_salida_gz,
    index=False,
    columns=COLUMNAS,
    encoding="utf-8-sig",
    compression={"method": "gzip", "mtime": 0},
    quoting=csv.QUOTE_ALL,
)

verificacion_gz = pd.read_csv(
    archivo_salida_gz,
    dtype=str,
    keep_default_na=False,
    encoding="utf-8-sig",
    compression="gzip",
)

if not verificacion_gz.equals(salida):
    raise ValueError(
        "La copia comprimida no coincide exactamente con la salida en memoria."
    )

validar_no_notacion_cientifica(
    verificacion_gz,
    "CSV.GZ AUTORITATIVO",
)

archivo_hash.write_text(
    hash_final + "\n",
    encoding="utf-8",
)

print("=== RESULTADO ===")
print("Archivo CSV:", archivo_salida)
print("Copia autoritativa:", archivo_salida_gz)
print("Archivo SHA-256:", archivo_hash)
print("Auditoría:", archivo_auditoria)
print("Revisión manual:", archivo_revision)
print()
print("Filas:", len(salida))
print("Columnas:", len(salida.columns))
print(
    "Títulos modificados:",
    int((entrada["Titulo"] != salida["Titulo"]).sum()),
)
print(
    "Keywords modificadas:",
    int((entrada["Keywords"] != salida["Keywords"]).sum()),
)
print(
    "Abstract modificados:",
    int((entrada["Abstract"] != salida["Abstract"]).sum()),
)
print(
    "Duplicados exactos conservados para fase posterior:",
    int(salida.duplicated().sum()),
)
print(
    "Grupos DOI repetidos conservados para fase posterior:",
    int((doi_counts > 1).sum()),
)
print()
print("Registros de datos en el CSV:", registros_datos)
print("Columnas físicas en el CSV:", 15)
print(
    "Nota: un visor puede mostrar 5315 líneas/registros contando "
    "también el encabezado. Los registros de DATOS son 5314."
)
print("SHA-256 autores_unam_limpios.csv:", hash_final)
print("CSV.GZ verificado:", True)
print()
print("Proceso terminado sin errores.")
